# 🏍️ ServisPro: Sistem Pakar Rekomendasi Servis Berkala Motor Matic
### Metode Inferensi: **Backward Chaining** (*goal-driven*)

**Kelompok 4 · Kelas R7M**

| No | Nama Lengkap | NPM |
|---|---|---|
| 1 | Muhamad Radi Taher | 202343501083 |
| 2 | Fatakhul Fatwa | 202343501026 |
| 3 | Sultan Rao Putra Mahmudin Batra | 202343501025 |
| 4 | Rangga Saputra | 202343501029 |

---

**Isi notebook:**
1. Penjelasan sistem & metode Backward Chaining
2. Basis pengetahuan (profil motor & interval servis, fakta, rule, goal)
3. Mesin inferensi Backward Chaining + lapisan urgensi
4. Pengujian dengan skenario demo
5. Pengujian *black box* beberapa skenario & beberapa profil motor
6. Cek satu servis saja (*goal* dipilih pengguna)
7. **Simpan basis pengetahuan** ke file `basis_pengetahuan_servis.sav`
8. Menjalankan aplikasi web `app.py` (Streamlit)
9. Kesimpulan

> Jalankan semua cell berurutan (**Run All**). File `.sav` yang dihasilkan dipakai oleh `app.py`.

## 1. Penjelasan Sistem

Sistem pakar ini merekomendasikan **jenis servis / penggantian komponen** motor matic berdasarkan jarak tempuh (km), lama pemakaian (bulan), dan keluhan yang dirasakan pengguna.

Sistem **tidak terikat pada satu tipe motor**. Pengguna memilih **profil motor** (Honda, Yamaha, Suzuki, atau *motor matic lain*), lalu sistem memakai **interval servis** dan **karakteristik** motor tersebut (injeksi/karburator, berradiator atau tidak, rem belakang cakram/tromol, tipe filter udara).

**Backward Chaining** adalah metode inferensi yang bekerja **mundur dari kesimpulan (goal) menuju fakta**:

1. Sistem mengambil satu **goal**, misalnya *Ganti Busi*.
2. Sistem mencari rule yang **THEN**-nya goal tersebut (misal R13 dan R14, hubungan **OR**).
3. Setiap **premis (IF)** dari rule dibuktikan satu per satu (hubungan **AND**):
   - Kalau premis adalah kesimpulan rule lain, premis itu jadi **sub-goal** dan dibuktikan secara rekursif.
   - Kalau premis adalah **fakta dasar**, sistem bertanya ke pengguna, **kecuali** fakta itu sudah diketahui dari profil motor.
   - Premis berawalan **NOT** terpenuhi kalau sub-goal-nya **tidak** terbukti.
4. Jika semua premis terpenuhi, goal **TERBUKTI**. Jika semua rule gagal, goal **TIDAK terbukti**.
5. Semua jawaban disimpan di **working memory** agar pertanyaan tidak diulang.

Kelebihannya: sistem **hanya menanyakan hal yang dibutuhkan** untuk membuktikan goal.

```
GOAL: Ganti Busi?
 └─ R14: IF indikasi_pengapian_lemah
      └─ SUB-GOAL: indikasi_pengapian_lemah?
           └─ R01: IF mesin_susah_hidup AND mesin_brebet
                ├─ FAKTA: mesin susah hidup?  -> tanya user
                └─ FAKTA: mesin brebet?       -> tanya user
```

In [1]:
import pickle
import pandas as pd

print("Library siap.")

Library siap.


## 2. Basis Pengetahuan (Knowledge Base)

### 2.1 Profil Motor & Interval Servis Berkala
Setiap profil motor punya **set interval sendiri** dan **karakteristik** yang langsung menjadi *fakta awal* di working memory (tidak ditanyakan lagi). Karakteristik bernilai `None` berarti berbeda antar varian, sehingga tetap ditanyakan ke pengguna.

**Sumber angka interval:**

| Profil | Sumber |
|---|---|
| Honda (BeAT/Scoopy/Genio, Vario 125, Vario/PCX/ADV 160) | Jadwal perawatan Buku Pedoman Pemilik Honda (pola item sama di BeAT, Click/Vario 125, ADV 160) dengan kelipatan servis AHM/KPB 4.000 km |
| Yamaha NMAX / Aerox / Lexi 155 | *Owner's Manual* Yamaha NMAX (GPD150-A), tabel *Periodic maintenance* |
| Yamaha 125 cc (Mio M3, Gear, Fazzio, Fino, FreeGo) | *Owner's Manual* Yamaha Gear 125 (LNP125) untuk busi, oli transmisi akhir, V-belt, minyak rem; sisanya anjuran bengkel resmi |
| Suzuki Address / Nex II | *Owner's Manual* Suzuki Address 110, *Maintenance Chart* |
| Motor matic lain | Nilai umum yang konservatif, bisa disesuaikan pengguna di aplikasi |

Nilai bertanda **[bengkel]** adalah rekomendasi umum bengkel karena tidak tercantum pasti di buku pedoman. Nilai `None` berarti interval itu **tidak berlaku** untuk motor tersebut (faktanya otomatis salah tanpa bertanya).

> Buku pedoman tiap negara/tahun bisa sedikit berbeda. Karena itu di aplikasi semua interval **bisa diubah** sesuai buku pedoman motor pengguna.

In [2]:
# 2a. Interval servis berkala
# INTERVAL_UMUM = nilai dasar yang konservatif (profil "motor matic lain").
# Setiap profil cukup menulis nilai yang BERBEDA dari nilai dasar ini.
# None = interval tidak berlaku untuk motor tersebut.
# ---------------------------------------------------------------------
INTERVAL_UMUM = {
    "oli_mesin_km":         4000,   # ganti oli mesin
    "oli_mesin_bulan":      3,      # [bengkel] maks. ±3 bulan untuk pemakaian harian
    "oli_gardan_km":        8000,   # ganti oli gardan / transmisi akhir
    "oli_gardan_bulan":     24,
    "busi_km":              8000,
    "filter_udara_km":      12000,  # ganti filter udara
    "filter_udara_debu_km": 6000,   # [bengkel] lebih cepat jika sering lewat jalan berdebu
    "filter_bersih_km":     4000,   # [bengkel] bersihkan filter (khusus tipe yang boleh dibersihkan)
    "servis_cvt_km":        8000,   # [bengkel] bersihkan CVT & cek roller
    "vbelt_km":             20000,
    "vbelt_gejala_km":      14000,  # [bengkel] ganti lebih awal jika CVT bergejala
    "injeksi_km":           8000,   # [bengkel] servis injektor/throttle body
    "karburator_km":        8000,   # [bengkel] servis karburator
    "minyak_rem_bulan":     24,
    "coolant_bulan":        24,
    "coolant_km":           None,
}

# Label & satuan tiap interval (dipakai tabel dan form penyesuaian di aplikasi)
LABEL_INTERVAL = {
    "oli_mesin_km":         ("Oli mesin", "km"),
    "oli_mesin_bulan":      ("Oli mesin", "bulan"),
    "oli_gardan_km":        ("Oli gardan", "km"),
    "oli_gardan_bulan":     ("Oli gardan", "bulan"),
    "busi_km":              ("Busi", "km"),
    "filter_udara_km":      ("Ganti filter udara", "km"),
    "filter_udara_debu_km": ("Ganti filter udara (jalan berdebu)", "km"),
    "filter_bersih_km":     ("Bersihkan filter udara", "km"),
    "servis_cvt_km":        ("Servis CVT", "km"),
    "vbelt_km":             ("V-belt", "km"),
    "vbelt_gejala_km":      ("V-belt (jika CVT bergejala)", "km"),
    "injeksi_km":           ("Servis injeksi", "km"),
    "karburator_km":        ("Servis karburator", "km"),
    "minyak_rem_bulan":     ("Minyak rem", "bulan"),
    "coolant_bulan":        ("Air radiator (coolant)", "bulan"),
    "coolant_km":           ("Air radiator (coolant)", "km"),
}

# Interval yang berasal dari rekomendasi bengkel di SEMUA profil
KUNCI_BENGKEL = {"oli_mesin_bulan", "filter_udara_debu_km", "filter_bersih_km", "servis_cvt_km",
                 "vbelt_gejala_km", "injeksi_km", "karburator_km"}

# Jadwal Honda (pola Buku Pedoman Pemilik + kelipatan servis AHM 4.000 km)
INTERVAL_HONDA = {
    "oli_mesin_km": 4000, "oli_gardan_km": 8000, "oli_gardan_bulan": 24, "busi_km": 8000,
    "filter_udara_km": 16000, "filter_udara_debu_km": 8000,
    "vbelt_km": 24000, "vbelt_gejala_km": 16000,
    "minyak_rem_bulan": 24, "coolant_bulan": 36,
}

# 2b. Profil motor
#   karakter : fakta awal (True/False). None = berbeda antar varian -> ditanyakan.
#   bengkel  : interval profil ini yang bersumber dari bengkel (selain KUNCI_BENGKEL).
# ---------------------------------------------------------------------
KARAKTERISTIK = ["motor_injeksi", "motor_berradiator", "rem_belakang_cakram", "filter_viscous"]

def profil(nama, sumber, interval, bengkel=(), **karakter):
    return {"nama": nama, "sumber": sumber, "interval": {**INTERVAL_UMUM, **interval},
            "bengkel": sorted(KUNCI_BENGKEL | set(bengkel)),
            "karakter": {k: karakter.get(k) for k in KARAKTERISTIK}}

PROFIL_MOTOR = {
    "honda_110": profil(
        "Honda BeAT / Scoopy / Genio (110 cc)",
        "Buku Pedoman Pemilik Honda BeAT + jadwal servis AHM",
        INTERVAL_HONDA, bengkel=["oli_gardan_km"],
        motor_injeksi=True, motor_berradiator=False, rem_belakang_cakram=False, filter_viscous=True),
    "honda_vario125": profil(
        "Honda Vario 125",
        "Buku Pedoman Pemilik Honda Vario 125 + jadwal servis AHM",
        INTERVAL_HONDA, bengkel=["oli_gardan_km"],
        motor_injeksi=True, motor_berradiator=True, rem_belakang_cakram=False, filter_viscous=True),
    "honda_160": profil(
        "Honda Vario 160 / PCX 160 / ADV 160 / Stylo 160",
        "Buku Pedoman Pemilik Honda ADV 160 + jadwal servis AHM",
        INTERVAL_HONDA, bengkel=["oli_gardan_km"],
        motor_injeksi=True, motor_berradiator=True, rem_belakang_cakram=None, filter_viscous=True),
    "yamaha_125": profil(
        "Yamaha Mio M3 / Gear 125 / Fazzio / Fino / FreeGo (125 cc)",
        "Owner's Manual Yamaha Gear 125 (LNP125) + anjuran bengkel resmi Yamaha",
        {"oli_mesin_km": 3000, "oli_gardan_km": 6000, "busi_km": 6000,
         "filter_udara_km": 12000, "filter_udara_debu_km": 6000, "servis_cvt_km": 12000,
         "vbelt_km": 25000, "vbelt_gejala_km": 16000, "injeksi_km": 10000, "minyak_rem_bulan": 24},
        bengkel=["oli_mesin_km", "oli_gardan_bulan", "filter_udara_km"],
        motor_injeksi=True, motor_berradiator=False, rem_belakang_cakram=False, filter_viscous=True),
    "yamaha_155": profil(
        "Yamaha NMAX / Aerox 155 / Lexi 155",
        "Owner's Manual Yamaha NMAX (GPD150-A)",
        {"oli_mesin_km": 4000, "oli_gardan_km": 12000, "busi_km": 8000,
         "filter_udara_km": 12000, "filter_udara_debu_km": 6000,
         "vbelt_km": 25000, "vbelt_gejala_km": 16000, "minyak_rem_bulan": 24,
         "coolant_km": 12000, "coolant_bulan": 36},
        bengkel=["oli_gardan_bulan", "coolant_bulan"],
        motor_injeksi=True, motor_berradiator=True, rem_belakang_cakram=None, filter_viscous=True),
    "suzuki_110": profil(
        "Suzuki Address / Nex II (113 cc)",
        "Owner's Manual Suzuki Address 110",
        {"oli_mesin_km": 4000, "oli_gardan_km": 8000, "busi_km": 8000,
         "filter_udara_km": 12000, "filter_udara_debu_km": 6000, "filter_bersih_km": 4000,
         "vbelt_km": 20000, "vbelt_gejala_km": 14000, "injeksi_km": 10000, "minyak_rem_bulan": 24},
        bengkel=["oli_gardan_bulan"],
        motor_injeksi=True, motor_berradiator=False, rem_belakang_cakram=False, filter_viscous=False),
    "umum": profil(
        "Motor matic lain (interval umum)",
        "Nilai umum yang konservatif; sesuaikan dengan buku pedoman motor Anda",
        {}, bengkel=list(INTERVAL_UMUM)),
}
PROFIL_DEMO = "honda_vario125"

# Tabel perbandingan interval antar profil
def teks_interval(v):
    return "-" if v is None else f"{v:,}".replace(",", ".")

tabel = pd.DataFrame({p["nama"].split(" (")[0]: {f"{LABEL_INTERVAL[k][0]} ({LABEL_INTERVAL[k][1]})": teks_interval(v)
                                                 for k, v in p["interval"].items()}
                      for p in PROFIL_MOTOR.values()})
tabel

,Honda BeAT / Scoopy / Genio,Honda Vario 125,Honda Vario 160 / PCX 160 / ADV 160 / Stylo 160,Yamaha Mio M3 / Gear 125 / Fazzio / Fino / FreeGo,Yamaha NMAX / Aerox 155 / Lexi 155,Suzuki Address / Nex II,Motor matic lain
Oli mesin (km),4.000,4.000,4.000,3.000,4.000,4.000,4.000
Oli mesin (bulan),3,3,3,3,3,3,3
Oli gardan (km),8.000,8.000,8.000,6.000,12.000,8.000,8.000
Oli gardan (bulan),24,24,24,24,24,24,24
Busi (km),8.000,8.000,8.000,6.000,8.000,8.000,8.000
Ganti filter udara (km),16.000,16.000,16.000,12.000,12.000,12.000,12.000
Ganti filter udara (jalan berdebu) (km),8.000,8.000,8.000,6.000,6.000,6.000,6.000
Bersihkan filter udara (km),4.000,4.000,4.000,4.000,4.000,4.000,4.000
Servis CVT (km),8.000,8.000,8.000,12.000,8.000,8.000,8.000
V-belt (km),24.000,24.000,24.000,25.000,25.000,20.000,20.000


### 2.2 Parameter Numerik
Pertanyaan angka (km / bulan). Setiap parameter **cukup ditanya sekali**, lalu hasilnya dipakai oleh beberapa fakta.

In [3]:
# 2c. Parameter numerik yang ditanyakan ke pengguna (disimpan sekali)
# ---------------------------------------------------------------------
PARAMETER = {
    "km_oli_mesin":     "Berapa km sejak terakhir ganti OLI MESIN?",
    "bln_oli_mesin":    "Berapa bulan sejak terakhir ganti OLI MESIN?",
    "km_oli_gardan":    "Berapa km sejak terakhir ganti OLI GARDAN?",
    "bln_oli_gardan":   "Berapa bulan sejak terakhir ganti OLI GARDAN?",
    "km_busi":          "Berapa km sejak terakhir ganti BUSI?",
    "km_filter":        "Berapa km sejak terakhir ganti FILTER UDARA?",
    "km_filter_bersih": "Berapa km sejak terakhir MEMBERSIHKAN FILTER UDARA?",
    "km_cvt":           "Berapa km sejak terakhir SERVIS/BERSIHKAN CVT?",
    "km_vbelt":         "Berapa km sejak terakhir ganti V-BELT?",
    "km_injeksi":       "Berapa km sejak terakhir SERVIS INJEKSI/THROTTLE BODY?",
    "km_karburator":    "Berapa km sejak terakhir SERVIS KARBURATOR?",
    "bln_minyak_rem":   "Berapa bulan sejak terakhir ganti MINYAK REM?",
    "bln_coolant":      "Berapa bulan sejak terakhir ganti AIR RADIATOR (coolant)?",
    "km_coolant":       "Berapa km sejak terakhir ganti AIR RADIATOR (coolant)?",
}

### 2.3 Fakta Dasar (*Askable Facts*)
- **numerik**: bernilai benar jika jawaban ≥ batas interval profil motor
- **ya_tidak**: kondisi fisik / keluhan yang ditanyakan langsung
- **karakteristik motor** (ya_tidak): sudah terisi dari profil motor; hanya ditanyakan jika profil tidak mengetahuinya

In [4]:
# 2d. Fakta dasar (askable facts)
#   - tipe "numerik"  : bernilai benar jika parameter >= batas interval
#   - tipe "ya_tidak" : ditanyakan langsung ke pengguna
# ---------------------------------------------------------------------
FAKTA_DASAR = {
    # --- fakta numerik (pengecekan interval)
    "km_oli_mesin_lewat":     {"tipe": "numerik", "param": "km_oli_mesin",     "batas": "oli_mesin_km"},
    "bln_oli_mesin_lewat":    {"tipe": "numerik", "param": "bln_oli_mesin",    "batas": "oli_mesin_bulan"},
    "km_oli_gardan_lewat":    {"tipe": "numerik", "param": "km_oli_gardan",    "batas": "oli_gardan_km"},
    "bln_oli_gardan_lewat":   {"tipe": "numerik", "param": "bln_oli_gardan",   "batas": "oli_gardan_bulan"},
    "km_busi_lewat":          {"tipe": "numerik", "param": "km_busi",          "batas": "busi_km"},
    "km_filter_lewat":        {"tipe": "numerik", "param": "km_filter",        "batas": "filter_udara_km"},
    "km_filter_debu_lewat":   {"tipe": "numerik", "param": "km_filter",        "batas": "filter_udara_debu_km"},
    "km_filter_bersih_lewat": {"tipe": "numerik", "param": "km_filter_bersih", "batas": "filter_bersih_km"},
    "km_cvt_lewat":           {"tipe": "numerik", "param": "km_cvt",           "batas": "servis_cvt_km"},
    "km_vbelt_lewat":         {"tipe": "numerik", "param": "km_vbelt",         "batas": "vbelt_km"},
    "km_vbelt_gejala_lewat":  {"tipe": "numerik", "param": "km_vbelt",         "batas": "vbelt_gejala_km"},
    "km_injeksi_lewat":       {"tipe": "numerik", "param": "km_injeksi",       "batas": "injeksi_km"},
    "km_karburator_lewat":    {"tipe": "numerik", "param": "km_karburator",    "batas": "karburator_km"},
    "bln_minyak_rem_lewat":   {"tipe": "numerik", "param": "bln_minyak_rem",   "batas": "minyak_rem_bulan"},
    "bln_coolant_lewat":      {"tipe": "numerik", "param": "bln_coolant",      "batas": "coolant_bulan"},
    "km_coolant_lewat":       {"tipe": "numerik", "param": "km_coolant",       "batas": "coolant_km"},

    # --- karakteristik motor (fakta awal dari profil; ditanya hanya jika belum diketahui)
    "motor_injeksi":       {"tipe": "ya_tidak", "tanya": "Apakah motor Anda bermesin injeksi (PGM-FI / FI), bukan karburator?"},
    "motor_berradiator":   {"tipe": "ya_tidak", "tanya": "Apakah motor Anda menggunakan pendingin radiator (cairan)?"},
    "rem_belakang_cakram": {"tipe": "ya_tidak", "tanya": "Apakah rem belakang motor Anda menggunakan cakram (disc brake)?"},
    "filter_viscous":      {"tipe": "ya_tidak", "tanya": "Apakah filter udara motor Anda tipe viscous/kertas berlapis oli (hanya boleh diganti, tidak boleh dibersihkan)?"},

    # --- fakta ya/tidak (kondisi fisik & keluhan)
    "indikator_oli_menyala": {"tipe": "ya_tidak", "tanya": "Apakah indikator ganti oli di panel meter menyala/berkedip?"},
    "oli_hitam_berkurang":   {"tipe": "ya_tidak", "tanya": "Saat dicek, apakah oli mesin hitam pekat atau di bawah batas minimum?"},
    "gardan_rembes":         {"tipe": "ya_tidak", "tanya": "Apakah ada rembesan oli di area gardan/roda belakang?"},
    "mesin_susah_hidup":     {"tipe": "ya_tidak", "tanya": "Apakah mesin susah dihidupkan?"},
    "mesin_brebet":          {"tipe": "ya_tidak", "tanya": "Apakah mesin brebet/tersendat saat digas?"},
    "sering_jalan_berdebu":  {"tipe": "ya_tidak", "tanya": "Apakah motor sering dipakai di jalan berdebu/proyek?"},
    "tarikan_berat":         {"tipe": "ya_tidak", "tanya": "Apakah tarikan motor terasa berat/loyo?"},
    "bunyi_cvt":             {"tipe": "ya_tidak", "tanya": "Apakah terdengar bunyi kasar/mendengung dari area CVT?"},
    "gredek":                {"tipe": "ya_tidak", "tanya": "Apakah motor bergetar (gredek) saat mulai jalan?"},
    "boros_bensin":          {"tipe": "ya_tidak", "tanya": "Apakah konsumsi bensin terasa lebih boros?"},
    "langsam_tidak_stabil":  {"tipe": "ya_tidak", "tanya": "Apakah putaran mesin saat langsam naik-turun/tidak stabil?"},
    "kampas_batas_aus":      {"tipe": "ya_tidak", "tanya": "Apakah kampas rem cakram sudah mencapai garis/indikator batas aus (±1 mm)?"},
    "tromol_batas_aus":      {"tipe": "ya_tidak", "tanya": "Apakah indikator keausan rem tromol belakang sudah di tanda batas, atau setelan rem belakang sudah habis?"},
    "rem_berbunyi":          {"tipe": "ya_tidak", "tanya": "Apakah rem berbunyi mencicit/gesekan logam saat dipakai?"},
    "rem_kurang_pakem":      {"tipe": "ya_tidak", "tanya": "Apakah rem terasa kurang pakem/jarak pengereman lebih jauh?"},
    "minyak_rem_keruh":      {"tipe": "ya_tidak", "tanya": "Apakah minyak rem di reservoir keruh/gelap atau di bawah garis LOWER?"},
    "mesin_overheat":        {"tipe": "ya_tidak", "tanya": "Apakah mesin sering overheat/terlalu panas?"},
    "coolant_berkurang":     {"tipe": "ya_tidak", "tanya": "Apakah air radiator di tabung cadangan sering berkurang?"},
}

### 2.4 Rule Base
Total **33 rule** berbentuk `IF premis AND premis THEN kesimpulan`. Beberapa rule dengan kesimpulan sama berarti hubungan **OR**. Premis `NOT x` terpenuhi jika `x` tidak terbukti.

| Kelompok | Rule | Kesimpulan |
|---|---|---|
| Sub-goal | R01–R05 | indikasi pengapian lemah, masalah CVT, sistem bahan bakar kotor, kampas habis |
| Oli mesin | R06–R09 | ganti_oli_mesin |
| Oli gardan | R10–R12 | ganti_oli_gardan |
| Busi | R13–R14 | ganti_busi |
| Filter udara | R15–R16 | ganti_filter_udara |
| Filter udara (bukan viscous) | R17 | bersihkan_filter_udara |
| CVT | R18–R19 | servis_cvt |
| V-belt | R20–R21 | ganti_vbelt |
| Injeksi (motor injeksi) | R22–R23 | servis_injeksi |
| Karburator (motor karburator) | R24–R25 | servis_karburator |
| Kampas rem | R26–R28 | ganti_kampas_rem |
| Minyak rem | R29–R30 | ganti_minyak_rem |
| Coolant (motor berradiator) | R31–R33 | ganti_coolant |

Rule yang memakai **karakteristik motor** (`motor_injeksi`, `motor_berradiator`, `rem_belakang_cakram`, `filter_viscous`) otomatis gugur untuk motor yang tidak cocok, tanpa menambah pertanyaan.

In [5]:
# 2e. Rule base (IF premis AND premis ... THEN kesimpulan)
#   Beberapa rule dengan kesimpulan yang sama = hubungan OR.
#   Premis "NOT x" terpenuhi jika x tidak terbukti.
# ---------------------------------------------------------------------
RULES = [
    # ---- Sub-goal (fakta antara)
    {"id": "R01", "if": ["mesin_susah_hidup", "mesin_brebet"],    "then": "indikasi_pengapian_lemah"},
    {"id": "R02", "if": ["tarikan_berat", "bunyi_cvt"],           "then": "indikasi_masalah_cvt"},
    {"id": "R03", "if": ["gredek"],                               "then": "indikasi_masalah_cvt"},
    {"id": "R04", "if": ["boros_bensin", "langsam_tidak_stabil"], "then": "indikasi_bbm_kotor"},
    {"id": "R05", "if": ["rem_berbunyi", "rem_kurang_pakem"],     "then": "indikasi_kampas_habis"},

    # ---- Goal: Ganti oli mesin
    {"id": "R06", "if": ["km_oli_mesin_lewat"],    "then": "ganti_oli_mesin"},
    {"id": "R07", "if": ["bln_oli_mesin_lewat"],   "then": "ganti_oli_mesin"},
    {"id": "R08", "if": ["indikator_oli_menyala"], "then": "ganti_oli_mesin"},
    {"id": "R09", "if": ["oli_hitam_berkurang"],   "then": "ganti_oli_mesin"},

    # ---- Goal: Ganti oli gardan
    {"id": "R10", "if": ["km_oli_gardan_lewat"],  "then": "ganti_oli_gardan"},
    {"id": "R11", "if": ["bln_oli_gardan_lewat"], "then": "ganti_oli_gardan"},
    {"id": "R12", "if": ["gardan_rembes"],        "then": "ganti_oli_gardan"},

    # ---- Goal: Ganti busi
    {"id": "R13", "if": ["km_busi_lewat"],            "then": "ganti_busi"},
    {"id": "R14", "if": ["indikasi_pengapian_lemah"], "then": "ganti_busi"},

    # ---- Goal: Ganti filter udara
    {"id": "R15", "if": ["km_filter_lewat"],                              "then": "ganti_filter_udara"},
    {"id": "R16", "if": ["sering_jalan_berdebu", "km_filter_debu_lewat"], "then": "ganti_filter_udara"},

    # ---- Goal: Bersihkan filter udara (hanya tipe yang boleh dibersihkan & belum waktunya ganti)
    {"id": "R17", "if": ["NOT filter_viscous", "NOT ganti_filter_udara", "km_filter_bersih_lewat"],
     "then": "bersihkan_filter_udara"},

    # ---- Goal: Servis/pembersihan CVT
    {"id": "R18", "if": ["km_cvt_lewat"],         "then": "servis_cvt"},
    {"id": "R19", "if": ["indikasi_masalah_cvt"], "then": "servis_cvt"},

    # ---- Goal: Ganti V-belt
    {"id": "R20", "if": ["km_vbelt_lewat"],                                "then": "ganti_vbelt"},
    {"id": "R21", "if": ["indikasi_masalah_cvt", "km_vbelt_gejala_lewat"], "then": "ganti_vbelt"},

    # ---- Goal: Servis injektor & throttle body (hanya motor injeksi)
    {"id": "R22", "if": ["motor_injeksi", "km_injeksi_lewat"],   "then": "servis_injeksi"},
    {"id": "R23", "if": ["motor_injeksi", "indikasi_bbm_kotor"], "then": "servis_injeksi"},

    # ---- Goal: Servis karburator (hanya motor karburator)
    {"id": "R24", "if": ["NOT motor_injeksi", "km_karburator_lewat"], "then": "servis_karburator"},
    {"id": "R25", "if": ["NOT motor_injeksi", "indikasi_bbm_kotor"],  "then": "servis_karburator"},

    # ---- Goal: Ganti kampas rem
    {"id": "R26", "if": ["kampas_batas_aus"],                            "then": "ganti_kampas_rem"},
    {"id": "R27", "if": ["indikasi_kampas_habis"],                       "then": "ganti_kampas_rem"},
    {"id": "R28", "if": ["NOT rem_belakang_cakram", "tromol_batas_aus"], "then": "ganti_kampas_rem"},

    # ---- Goal: Ganti minyak rem
    {"id": "R29", "if": ["bln_minyak_rem_lewat"], "then": "ganti_minyak_rem"},
    {"id": "R30", "if": ["minyak_rem_keruh"],     "then": "ganti_minyak_rem"},

    # ---- Goal: Ganti air radiator (hanya motor berradiator)
    {"id": "R31", "if": ["motor_berradiator", "bln_coolant_lewat"],                   "then": "ganti_coolant"},
    {"id": "R32", "if": ["motor_berradiator", "km_coolant_lewat"],                    "then": "ganti_coolant"},
    {"id": "R33", "if": ["motor_berradiator", "mesin_overheat", "coolant_berkurang"], "then": "ganti_coolant"},
]

### 2.5 Daftar Goal (Jenis Servis)

In [6]:
# 2f. Daftar GOAL beserta deskripsi tindakan servis
# ---------------------------------------------------------------------
GOALS = {
    "ganti_oli_mesin":        ("Ganti Oli Mesin", "Ganti oli sesuai spesifikasi buku pedoman (mis. SAE 10W-30/10W-40, JASO MB) dan bersihkan saringan kasa oli."),
    "ganti_oli_gardan":       ("Ganti Oli Gardan (Final Drive)", "Ganti oli gardan sesuai takaran pabrikan (±100-150 ml), cek seal bila rembes."),
    "ganti_busi":             ("Ganti Busi", "Ganti busi sesuai tipe pabrikan, cek celah elektroda dan kondisi koil."),
    "ganti_filter_udara":     ("Ganti Filter Udara", "Ganti elemen saringan udara (tipe viscous/kertas berlapis oli tidak boleh dibersihkan dengan semprotan angin)."),
    "bersihkan_filter_udara": ("Bersihkan Filter Udara", "Bersihkan elemen saringan udara sesuai petunjuk buku pedoman; ganti bila sobek atau terlalu kotor."),
    "servis_cvt":             ("Servis / Pembersihan CVT", "Bersihkan debu kampas di ruang CVT, periksa roller, slider, dan mangkok kopling."),
    "ganti_vbelt":            ("Ganti V-Belt", "Ganti V-belt; periksa juga roller dan pulley karena biasanya aus bersamaan."),
    "servis_injeksi":         ("Servis Injektor & Throttle Body", "Bersihkan injektor dan throttle body dengan injector cleaner, lalu reset ECU."),
    "servis_karburator":      ("Servis Karburator", "Bersihkan karburator (spuyer & ruang pelampung), lalu setel campuran udara-bensin dan putaran langsam."),
    "ganti_kampas_rem":       ("Ganti Kampas Rem", "Ganti kampas rem depan/belakang, periksa permukaan cakram/tromol."),
    "ganti_minyak_rem":       ("Ganti Minyak Rem", "Kuras dan ganti minyak rem (DOT 3/DOT 4), lalu buang angin (bleeding)."),
    "ganti_coolant":          ("Ganti Air Radiator (Coolant)", "Kuras dan isi ulang coolant, periksa selang dan tutup radiator."),
}

NAMA_FAKTA_ANTARA = {
    "indikasi_pengapian_lemah": "Indikasi pengapian lemah",
    "indikasi_masalah_cvt":     "Indikasi masalah pada CVT",
    "indikasi_bbm_kotor":       "Indikasi sistem bahan bakar kotor",
    "indikasi_kampas_habis":    "Indikasi kampas rem habis",
}

# Parameter lapisan urgensi (dipakai setelah inferensi selesai)
URGENSI = {"segera": 1.5, "dekat": 0.8}   # rasio nilai / batas interval
GOAL_KESELAMATAN = {"ganti_kampas_rem"}   # goal terbukti = selalu SEGERA

print(f"Basis pengetahuan dimuat: {len(PROFIL_MOTOR)} profil motor, {len(GOALS)} goal, {len(RULES)} rule, "
      f"{len(FAKTA_DASAR)} fakta dasar, {len(PARAMETER)} parameter numerik")

Basis pengetahuan dimuat: 7 profil motor, 12 goal, 33 rule, 38 fakta dasar, 14 parameter numerik


## 3. Mesin Inferensi Backward Chaining

Inti metode ada di fungsi **`_buktikan()`** yang bersifat **rekursif**. Langkah 1–7 diberi komentar di dalam kode:

| Langkah | Proses |
|---|---|
| 1 | Cek *working memory* (termasuk **fakta awal dari profil motor**), jika sudah diketahui langsung dipakai |
| 2 | Cari rule yang THEN-nya = goal |
| 3 | Tidak ada rule → fakta dasar → **tanya user** |
| 4 | **Iterasi** setiap rule kandidat (OR) |
| 5 | Setiap premis jadi **sub-goal**, dibuktikan rekursif (AND); premis `NOT` dibalik |
| 6 | Semua premis benar → goal **TERBUKTI**, iterasi berhenti |
| 7 | Semua rule gagal → goal **TIDAK terbukti** |

Fungsi **`jelaskan()`** adalah *explanation facility* untuk menjawab pertanyaan "kenapa servis ini direkomendasikan?".

**Lapisan urgensi** (`tingkat_urgensi()`) dijalankan **setelah** inferensi selesai dan tidak menambah pertanyaan:

| Status | Arti |
|---|---|
| **SEGERA** | Goal terbukti dan nilainya sudah ≥ 150% batas interval, atau menyangkut keselamatan (kampas rem) |
| **PERLU** | Goal terbukti |
| **DEKAT** | Goal belum terbukti, tapi sudah ≥ 80% batas interval |
| **AMAN** | Goal belum terbukti |

In [7]:
class BackwardChaining:
    def __init__(self, rules, fakta_dasar, parameter, interval, penjawab, fakta_awal=None):
        self.rules = rules
        self.fakta_dasar = fakta_dasar
        self.parameter = parameter
        self.interval = interval        # interval servis milik profil motor yang dipilih
        self.penjawab = penjawab        # objek yang bertugas menanyai pengguna
        self.fakta_awal = dict(fakta_awal or {})   # fakta yang sudah diketahui dari profil motor
        self.memori = dict(self.fakta_awal)        # working memory: fakta -> True/False
        self.nilai_param = {}           # cache jawaban numerik
        self.bukti = {}                 # goal -> rule yang berhasil membuktikannya
        self.trace = []                 # catatan proses inferensi
        self.jalur = []                 # jalur penelusuran aktif: goal -> rule -> premis ...
        self.goal_aktif = None          # goal utama yang sedang dibuktikan

    def log(self, depth, teks):
        self.trace.append("    " * depth + teks)

    # -----------------------------------------------------------------
    # Fungsi inti backward chaining (rekursif)
    # -----------------------------------------------------------------
    def buktikan(self, goal, depth=0):
        # Catat goal ke jalur penelusuran (dipakai GUI untuk visualisasi)
        self.jalur.append(goal)
        hasil = self._buktikan(goal, depth)
        self.jalur.pop()
        return hasil

    def _buktikan(self, goal, depth):
        # [LANGKAH 1] Cek working memory: jika goal sudah diketahui (dari
        # profil motor atau sudah pernah dibuktikan), pakai hasilnya agar
        # tidak bertanya dua kali.
        if goal in self.memori:
            asal = "diketahui dari profil motor" if goal in self.fakta_awal else "sudah diketahui"
            self.log(depth, f"- '{goal}' {asal} = {self.memori[goal]}")
            return self.memori[goal]

        # [LANGKAH 2] Cari semua rule yang KESIMPULANNYA (THEN) adalah goal.
        rule_kandidat = [r for r in self.rules if r["then"] == goal]

        # [LANGKAH 3] Tidak ada rule yang menyimpulkan goal -> goal adalah
        # FAKTA DASAR, sehingga nilainya diperoleh dengan bertanya ke user.
        if not rule_kandidat:
            hasil = self.tanya_fakta(goal)
            self.memori[goal] = hasil
            info = self.fakta_dasar[goal]
            catatan = (" (interval tidak berlaku untuk motor ini)"
                       if info["tipe"] == "numerik" and self.interval.get(info["batas"]) is None else "")
            self.log(depth, f"- Fakta dasar '{goal}' = {hasil}{catatan}")
            return hasil

        # [LANGKAH 4] ITERASI BACKWARD CHAINING:
        # Coba setiap rule kandidat satu per satu (hubungan OR).
        for rule in rule_kandidat:
            self.log(depth, f"> Mencoba {rule['id']}: IF {' AND '.join(rule['if'])} THEN {goal}")
            terpenuhi = True
            self.jalur.append(rule["id"])

            # [LANGKAH 5] Setiap premis menjadi SUB-GOAL baru dan dibuktikan
            # secara rekursif (mundur dari kesimpulan menuju fakta).
            # Premis berawalan "NOT " terpenuhi jika sub-goal TIDAK terbukti.
            for premis in rule["if"]:
                negasi = premis.startswith("NOT ")
                nilai = self.buktikan(premis[4:] if negasi else premis, depth + 1)
                if nilai == negasi:
                    # Satu premis gagal -> rule gagal (hubungan AND),
                    # premis berikutnya tidak perlu ditanyakan.
                    terpenuhi = False
                    self.log(depth, f"x {rule['id']} gagal (premis '{premis}' tidak terpenuhi)")
                    break
            self.jalur.pop()

            # [LANGKAH 6] Semua premis terbukti -> goal TERBUKTI.
            # Iterasi dihentikan, rule kandidat lain tidak perlu dicoba.
            if terpenuhi:
                self.memori[goal] = True
                self.bukti[goal] = rule
                self.log(depth, f"✓ {rule['id']} berhasil -> '{goal}' TERBUKTI")
                return True

        # [LANGKAH 7] Semua rule kandidat gagal -> goal TIDAK terbukti.
        self.memori[goal] = False
        self.log(depth, f"x Semua rule untuk '{goal}' gagal -> TIDAK terbukti")
        return False

    # -----------------------------------------------------------------
    # Mengambil nilai fakta dasar dari pengguna
    # -----------------------------------------------------------------
    def tanya_fakta(self, fakta):
        info = self.fakta_dasar[fakta]
        if info["tipe"] == "ya_tidak":
            return self.penjawab.ya_tidak(fakta, info["tanya"])

        # Fakta numerik: dibandingkan dengan batas interval profil motor.
        # Batas None = interval tidak berlaku untuk motor ini, jadi fakta
        # langsung bernilai salah TANPA bertanya ke pengguna.
        batas = self.interval.get(info["batas"])
        if batas is None:
            return False

        # Parameter cukup ditanya sekali, lalu dipakai ulang.
        param = info["param"]
        if param not in self.nilai_param:
            self.nilai_param[param] = self.penjawab.angka(param, self.parameter[param])
        return self.nilai_param[param] >= batas

    # -----------------------------------------------------------------
    # Jalankan backward chaining untuk daftar goal yang diminta
    # (semua goal, atau hanya goal tertentu yang dipilih pengguna)
    # -----------------------------------------------------------------
    def jalankan(self, daftar_goal):
        hasil = []
        for goal in daftar_goal:
            self.goal_aktif = goal
            self.log(0, f"\n=== GOAL: {goal} ===")
            if self.buktikan(goal):
                hasil.append(goal)
        self.goal_aktif = None
        return hasil

    # -----------------------------------------------------------------
    # Penjelasan (explanation facility): kenapa goal direkomendasikan
    # -----------------------------------------------------------------
    def jelaskan(self, goal):
        rule = self.bukti.get(goal)
        if not rule:
            return []
        alasan = []
        for premis in rule["if"]:
            negasi = premis.startswith("NOT ")
            inti = premis[4:] if negasi else premis
            info = self.fakta_dasar.get(inti, {})
            if inti in self.bukti and not negasi:          # premis = sub-goal
                alasan.append(f"{NAMA_FAKTA_ANTARA.get(inti, inti)} ({self.bukti[inti]['id']}):")
                alasan += ["    > " + a for a in self.jelaskan(inti)]
            elif info.get("tipe") == "numerik":
                nilai = self.nilai_param[info["param"]]
                batas = self.interval[info["batas"]]
                satuan = "bulan" if info["param"].startswith("bln") else "km"
                komponen = self.parameter[info["param"]].split("terakhir ")[1].rstrip("?")
                alasan.append(f"Sudah {nilai:,} {satuan} sejak terakhir {komponen.lower()} "
                              f"(batas interval {batas:,} {satuan})".replace(",", "."))
            elif info.get("tipe") == "ya_tidak":
                asal = "Profil motor - " if inti in self.fakta_awal else ""
                alasan.append(f"{asal}{'Tidak' if negasi else 'Ya'}: {info['tanya']}")
            else:                                          # NOT sub-goal / NOT goal
                nama = GOALS[inti][0] if inti in GOALS else NAMA_FAKTA_ANTARA.get(inti, inti)
                alasan.append(f"Belum terbukti: {nama}")
        return alasan


# ---------------------------------------------------------------------
# Lapisan urgensi: dihitung SETELAH inferensi selesai, dari fakta numerik
# yang sudah ada di working memory (tidak menambah pertanyaan).
#   SEGERA : goal terbukti dan sudah jauh melewati batas / menyangkut keselamatan
#   PERLU  : goal terbukti
#   DEKAT  : goal belum terbukti, tapi sudah >= 80% dari batas interval
#   AMAN   : goal belum terbukti
# ---------------------------------------------------------------------
def rasio_bukti(engine, goal):
    """Rasio nilai/batas semua fakta numerik pada rantai pembuktian goal."""
    hasil = []
    for premis in engine.bukti[goal]["if"] if goal in engine.bukti else []:
        info = engine.fakta_dasar.get(premis, {})
        if premis in engine.bukti:
            hasil += rasio_bukti(engine, premis)
        elif info.get("tipe") == "numerik":
            hasil.append(engine.nilai_param[info["param"]] / engine.interval[info["batas"]])
    return hasil


def tingkat_urgensi(engine, goal):
    """Return (kode, keterangan) untuk satu goal."""
    if engine.memori.get(goal):
        if goal in GOAL_KESELAMATAN:
            return "SEGERA", "Menyangkut keselamatan berkendara"
        rasio = max(rasio_bukti(engine, goal), default=0)
        if rasio >= URGENSI["segera"]:
            return "SEGERA", f"Sudah {rasio * 100:.0f}% dari batas interval"
        return "PERLU", ""

    # Goal belum terbukti: cari fakta numerik yang hampir mencapai batas,
    # hanya pada rule yang premis lainnya sudah terpenuhi.
    terbaik = None
    for rule in engine.rules:
        if rule["then"] != goal:
            continue
        for premis in rule["if"]:
            info = engine.fakta_dasar.get(premis, {})
            if info.get("tipe") != "numerik":
                continue
            batas = engine.interval.get(info["batas"])
            nilai = engine.nilai_param.get(info["param"])
            if batas is None or nilai is None:
                continue
            lain = [p for p in rule["if"] if p != premis]
            if not all(engine.memori.get(p[4:]) is False if p.startswith("NOT ")
                       else engine.memori.get(p) is True for p in lain):
                continue
            rasio = nilai / batas
            if URGENSI["dekat"] <= rasio < 1 and (terbaik is None or rasio > terbaik[0]):
                satuan = "bulan" if info["param"].startswith("bln") else "km"
                terbaik = (rasio, f"Kurang {batas - nilai:,} {satuan} lagi (batas {batas:,} {satuan})".replace(",", "."))
    if terbaik:
        return "DEKAT", terbaik[1]
    return "AMAN", ""

## 4. Pengujian dengan Skenario Demo

Jawaban pengguna diisi otomatis (`JAWABAN_DEMO`) dengan profil **Honda Vario 125** supaya proses inferensinya bisa langsung dilihat. Perhatikan:
- Pertanyaan **bulan oli mesin** muncul karena R06 (km) gagal, lalu sistem mencoba R07.
- Pertanyaan **indikator oli** tidak pernah muncul karena R07 sudah berhasil.
- Karakteristik motor (**injeksi, radiator, rem belakang, tipe filter**) tidak ditanyakan karena sudah diketahui dari profil.
- Goal **Servis Karburator** dan **Bersihkan Filter Udara** langsung gugur dari fakta profil, tanpa pertanyaan.

In [8]:
class PenjawabKonsol:
    """Menanyakan fakta ke pengguna lewat terminal."""

    def ya_tidak(self, fakta, pertanyaan):
        while True:
            jwb = input(f"  [?] {pertanyaan} (y/t): ").strip().lower()
            if jwb in ("y", "ya"):
                return True
            if jwb in ("t", "tidak", "n"):
                return False
            print("      Jawab dengan 'y' atau 't'.")

    def angka(self, param, pertanyaan):
        while True:
            jwb = input(f"  [?] {pertanyaan} ").strip().replace(".", "")
            if jwb.isdigit():
                return int(jwb)
            print("      Masukkan angka bulat, contoh: 4500")


class PenjawabDemo:
    """Jawaban otomatis untuk demo/pengujian (tanpa input manual)."""

    def __init__(self, jawaban):
        self.jawaban = jawaban

    def ya_tidak(self, fakta, pertanyaan):
        nilai = self.jawaban.get(fakta, False)
        print(f"  [?] {pertanyaan} (y/t): {'y' if nilai else 't'}")
        return nilai

    def angka(self, param, pertanyaan):
        nilai = self.jawaban.get(param, 0)
        print(f"  [?] {pertanyaan} {nilai}")
        return nilai


def buat_engine(kode_profil, penjawab):
    """Buat mesin inferensi untuk satu profil motor (interval + fakta awal)."""
    p = PROFIL_MOTOR[kode_profil]
    fakta_awal = {k: v for k, v in p["karakter"].items() if v is not None}
    return BackwardChaining(RULES, FAKTA_DASAR, PARAMETER, p["interval"], penjawab, fakta_awal)


JAWABAN_DEMO = {
    "km_oli_mesin": 3200, "bln_oli_mesin": 4,
    "km_oli_gardan": 7000, "bln_oli_gardan": 10,
    "km_busi": 6000, "mesin_susah_hidup": True, "mesin_brebet": True,
    "km_filter": 9000, "sering_jalan_berdebu": True,
    "km_cvt": 5000, "tarikan_berat": True, "bunyi_cvt": True,
    "km_vbelt": 17000,
    "km_injeksi": 6000, "boros_bensin": False,
    "kampas_batas_aus": False, "rem_berbunyi": True, "rem_kurang_pakem": False, "tromol_batas_aus": False,
    "bln_minyak_rem": 40, "minyak_rem_keruh": False,
    "bln_coolant": 12, "mesin_overheat": False,
    # hanya terpakai jika profil motor tidak mengetahui karakteristiknya
    "motor_injeksi": True, "motor_berradiator": True, "rem_belakang_cakram": False, "filter_viscous": True,
}


def tampilkan_hasil(engine, rekomendasi, tampil_trace=True):
    garis = "=" * 65
    if tampil_trace:
        print(f"\n{garis}\n PROSES INFERENSI BACKWARD CHAINING\n{garis}")
        print("\n".join(engine.trace))

    print(f"\n{garis}\n HASIL REKOMENDASI SERVIS\n{garis}")
    if not rekomendasi:
        print("Motor dalam kondisi baik. Belum ada servis yang perlu dilakukan.")
    for i, goal in enumerate(rekomendasi, 1):
        nama, tindakan = GOALS[goal]
        kode, ket = tingkat_urgensi(engine, goal)
        print(f"\n{i}. {nama}  [{engine.bukti[goal]['id']}]  <{kode}{': ' + ket if ket else ''}>")
        print(f"   Tindakan : {tindakan}")
        print("   Alasan   :")
        for a in engine.jelaskan(goal):
            print(f"     {'' if a.startswith('    >') else '- '}{a}")

    dekat = [(g, tingkat_urgensi(engine, g)) for g in engine.memori if g in GOALS and not engine.memori[g]]
    dekat = [(g, u) for g, u in dekat if u[0] == "DEKAT"]
    if dekat:
        print("\nSebentar lagi jatuh tempo:")
        for g, (_, ket) in dekat:
            print(f"  ~ {GOALS[g][0]}: {ket}")
    tidak = [GOALS[g][0] for g in engine.memori if g in GOALS and not engine.memori[g]]
    if tidak:
        print(f"\nBelum perlu: {', '.join(tidak)}")
    print("\n* Rekomendasi bersifat panduan. Ikuti buku pedoman pemilik motor Anda.")

In [9]:
print(f"Profil motor: {PROFIL_MOTOR[PROFIL_DEMO]['nama']}\n")
engine_demo = buat_engine(PROFIL_DEMO, PenjawabDemo(JAWABAN_DEMO))
rekomendasi_demo = engine_demo.jalankan(list(GOALS))
tampilkan_hasil(engine_demo, rekomendasi_demo)

Profil motor: Honda Vario 125

  [?] Berapa km sejak terakhir ganti OLI MESIN? 3200
  [?] Berapa bulan sejak terakhir ganti OLI MESIN? 4
  [?] Berapa km sejak terakhir ganti OLI GARDAN? 7000
  [?] Berapa bulan sejak terakhir ganti OLI GARDAN? 10
  [?] Apakah ada rembesan oli di area gardan/roda belakang? (y/t): t
  [?] Berapa km sejak terakhir ganti BUSI? 6000
  [?] Apakah mesin susah dihidupkan? (y/t): y
  [?] Apakah mesin brebet/tersendat saat digas? (y/t): y
  [?] Berapa km sejak terakhir ganti FILTER UDARA? 9000
  [?] Apakah motor sering dipakai di jalan berdebu/proyek? (y/t): y
  [?] Berapa km sejak terakhir SERVIS/BERSIHKAN CVT? 5000
  [?] Apakah tarikan motor terasa berat/loyo? (y/t): y
  [?] Apakah terdengar bunyi kasar/mendengung dari area CVT? (y/t): y
  [?] Berapa km sejak terakhir ganti V-BELT? 17000
  [?] Berapa km sejak terakhir SERVIS INJEKSI/THROTTLE BODY? 6000
  [?] Apakah konsumsi bensin terasa lebih boros? (y/t): t
  [?] Apakah kampas rem cakram sudah mencapai garis/

> 💡 **Mode teks interaktif (opsional):** hapus tanda `#` di cell berikut untuk konsultasi lewat `input()`. Ganti `"umum"` dengan kode profil lain bila perlu.

In [10]:
# engine_teks = buat_engine("umum", PenjawabKonsol())
# tampilkan_hasil(engine_teks, engine_teks.jalankan(list(GOALS)))

## 5. Pengujian *Black Box*

Beberapa skenario kondisi motor diuji pada **profil motor yang berbeda** untuk memastikan hasil sistem sesuai dengan harapan pakar. Kolom **Jumlah Pertanyaan** menunjukkan efisiensi backward chaining dibandingkan jika semua fakta ditanyakan.

Skenario 9 dan 10 memakai jawaban yang **sama persis** tetapi profil berbeda, untuk membuktikan interval mengikuti motor yang dipilih.

In [11]:
class PenjawabDiam:
    """Penjawab otomatis tanpa output. Fakta yang tidak diisi = Tidak / 0."""
    def __init__(self, jawaban):
        self.jawaban, self.ditanya = jawaban, 0
    def ya_tidak(self, fakta, pertanyaan):
        self.ditanya += 1
        return self.jawaban.get(fakta, False)
    def angka(self, param, pertanyaan):
        self.ditanya += 1
        return self.jawaban.get(param, 0)


SKENARIO = [
    ("Motor baru, tidak ada keluhan", "honda_110", {}, []),
    ("Oli mesin sudah 4.500 km", "honda_vario125", {"km_oli_mesin": 4500}, ["ganti_oli_mesin"]),
    ("Oli mesin 2.000 km tapi sudah 4 bulan", "honda_vario125", {"km_oli_mesin": 2000, "bln_oli_mesin": 4}, ["ganti_oli_mesin"]),
    ("Mesin susah hidup & brebet", "honda_110", {"mesin_susah_hidup": True, "mesin_brebet": True}, ["ganti_busi"]),
    ("Motor gredek, V-belt 17.000 km", "yamaha_155", {"gredek": True, "km_vbelt": 17000}, ["servis_cvt", "ganti_vbelt"]),
    ("Rem bunyi & kurang pakem", "honda_110", {"rem_berbunyi": True, "rem_kurang_pakem": True}, ["ganti_kampas_rem"]),
    ("Indikator tromol belakang di batas aus", "honda_110", {"tromol_batas_aus": True}, ["ganti_kampas_rem"]),
    ("Overheat tapi motor tanpa radiator", "honda_110", {"mesin_overheat": True, "coolant_berkurang": True}, []),
    ("Oli mesin 3.500 km (batas Honda 4.000)", "honda_110", {"km_oli_mesin": 3500}, []),
    ("Oli mesin 3.500 km (batas Yamaha 125 = 3.000)", "yamaha_125", {"km_oli_mesin": 3500}, ["ganti_oli_mesin"]),
    ("Coolant sudah 13.000 km", "yamaha_155", {"km_coolant": 13000}, ["ganti_coolant"]),
    ("Filter udara belum dibersihkan 5.000 km", "suzuki_110", {"km_filter": 5000, "km_filter_bersih": 5000}, ["bersihkan_filter_udara"]),
    ("Motor karburator, servis terakhir 9.000 km", "umum", {"motor_injeksi": False, "km_karburator": 9000}, ["servis_karburator"]),
    ("Servis besar 24.000 km", "honda_vario125",
     {"km_oli_mesin": 24000, "km_oli_gardan": 24000, "km_busi": 24000, "km_filter": 24000,
      "km_cvt": 24000, "km_vbelt": 24000, "km_injeksi": 24000},
     ["ganti_oli_mesin", "ganti_oli_gardan", "ganti_busi", "ganti_filter_udara",
      "servis_cvt", "ganti_vbelt", "servis_injeksi"]),
]

TOTAL_KEMUNGKINAN = len(PARAMETER) + sum(1 for f in FAKTA_DASAR.values() if f["tipe"] == "ya_tidak")
baris = []
for i, (nama, kode_profil, jawaban, harapan) in enumerate(SKENARIO, 1):
    pj = PenjawabDiam(jawaban)
    eng = buat_engine(kode_profil, pj)
    hasil = eng.jalankan(list(GOALS))
    baris.append({
        "No": i,
        "Skenario": nama,
        "Profil": PROFIL_MOTOR[kode_profil]["nama"].split(" (")[0],
        "Hasil Sistem": ", ".join(f"{GOALS[g][0]} ({eng.bukti[g]['id']})" for g in hasil) or "Tidak ada servis",
        "Jumlah Pertanyaan": f"{pj.ditanya} dari {TOTAL_KEMUNGKINAN}",
        "Status": "✅ Sesuai" if sorted(hasil) == sorted(harapan) else "❌ Tidak sesuai",
    })

df_uji = pd.DataFrame(baris)
sesuai = (df_uji["Status"] == "✅ Sesuai").sum()
print(f"Akurasi pengujian: {sesuai}/{len(df_uji)} skenario sesuai ({sesuai / len(df_uji) * 100:.0f}%)")
df_uji

Akurasi pengujian: 14/14 skenario sesuai (100%)


,No,Skenario,Profil,Hasil Sistem,Jumlah Pertanyaan,Status
0,1,"Motor baru, tidak ada keluhan",Honda BeAT / Scoopy / Genio,Tidak ada servis,22 dari 36,✅ Sesuai
1,2,Oli mesin sudah 4.500 km,Honda Vario 125,Ganti Oli Mesin (R06),21 dari 36,✅ Sesuai
2,3,Oli mesin 2.000 km tapi sudah 4 bulan,Honda Vario 125,Ganti Oli Mesin (R07),22 dari 36,✅ Sesuai
3,4,Mesin susah hidup & brebet,Honda BeAT / Scoopy / Genio,Ganti Busi (R14),23 dari 36,✅ Sesuai
4,5,"Motor gredek, V-belt 17.000 km",Yamaha NMAX / Aerox 155 / Lexi 155,"Servis / Pembersihan CVT (R19), Ganti V-Belt (...",26 dari 36,✅ Sesuai
5,6,Rem bunyi & kurang pakem,Honda BeAT / Scoopy / Genio,Ganti Kampas Rem (R27),22 dari 36,✅ Sesuai
6,7,Indikator tromol belakang di batas aus,Honda BeAT / Scoopy / Genio,Ganti Kampas Rem (R28),22 dari 36,✅ Sesuai
7,8,Overheat tapi motor tanpa radiator,Honda BeAT / Scoopy / Genio,Tidak ada servis,22 dari 36,✅ Sesuai
8,9,Oli mesin 3.500 km (batas Honda 4.000),Honda BeAT / Scoopy / Genio,Tidak ada servis,22 dari 36,✅ Sesuai
9,10,Oli mesin 3.500 km (batas Yamaha 125 = 3.000),Yamaha Mio M3 / Gear 125 / Fazzio / Fino / FreeGo,Ganti Oli Mesin (R06),19 dari 36,✅ Sesuai


## 6. Cek Satu Servis Saja

Karena backward chaining bersifat *goal-driven*, pengguna bisa memilih **goal tertentu saja**. Sistem hanya menelusuri rule menuju goal itu, sehingga pertanyaannya jauh lebih sedikit. Contoh: pengguna Yamaha NMAX hanya ingin tahu *"V-belt saya perlu diganti atau belum?"*

In [12]:
pj = PenjawabDiam({"km_vbelt": 21000, "gredek": True})
eng = buat_engine("yamaha_155", pj)
hasil = eng.jalankan(["ganti_vbelt"])          # hanya satu goal

print("\n".join(eng.trace))
print(f"\nJumlah pertanyaan: {pj.ditanya} (konsultasi lengkap bisa sampai {TOTAL_KEMUNGKINAN})")
for g in ["ganti_vbelt"]:
    kode, ket = tingkat_urgensi(eng, g)
    print(f"{GOALS[g][0]}: {kode} {ket}")

# Contoh status DEKAT: V-belt 21.000 km tanpa gejala CVT (batas 25.000 km)
pj2 = PenjawabDiam({"km_vbelt": 21000})
eng2 = buat_engine("yamaha_155", pj2)
eng2.jalankan(["ganti_vbelt"])
print("Tanpa gejala CVT ->", *tingkat_urgensi(eng2, "ganti_vbelt"))


=== GOAL: ganti_vbelt ===
> Mencoba R20: IF km_vbelt_lewat THEN ganti_vbelt
    - Fakta dasar 'km_vbelt_lewat' = False
x R20 gagal (premis 'km_vbelt_lewat' tidak terpenuhi)
> Mencoba R21: IF indikasi_masalah_cvt AND km_vbelt_gejala_lewat THEN ganti_vbelt
    > Mencoba R02: IF tarikan_berat AND bunyi_cvt THEN indikasi_masalah_cvt
        - Fakta dasar 'tarikan_berat' = False
    x R02 gagal (premis 'tarikan_berat' tidak terpenuhi)
    > Mencoba R03: IF gredek THEN indikasi_masalah_cvt
        - Fakta dasar 'gredek' = True
    ✓ R03 berhasil -> 'indikasi_masalah_cvt' TERBUKTI
    - Fakta dasar 'km_vbelt_gejala_lewat' = True
✓ R21 berhasil -> 'ganti_vbelt' TERBUKTI

Jumlah pertanyaan: 3 (konsultasi lengkap bisa sampai 36)
Ganti V-Belt: PERLU 
Tanpa gejala CVT -> DEKAT Kurang 4.000 km lagi (batas 25.000 km)


## 7. Simpan Basis Pengetahuan

Seluruh basis pengetahuan (profil motor & interval, parameter, fakta, rule, goal) disimpan ke file **`basis_pengetahuan_servis.sav`** menggunakan `pickle`.
File ini kemudian di-*load* oleh aplikasi web `app.py`, sama seperti model machine learning yang disimpan lalu dipakai aplikasi.

In [13]:
basis_pengetahuan = {
    "PROFIL_MOTOR": PROFIL_MOTOR,
    "PROFIL_DEMO": PROFIL_DEMO,
    "KARAKTERISTIK": KARAKTERISTIK,
    "LABEL_INTERVAL": LABEL_INTERVAL,
    "PARAMETER": PARAMETER,
    "FAKTA_DASAR": FAKTA_DASAR,
    "RULES": RULES,
    "GOALS": GOALS,
    "NAMA_FAKTA_ANTARA": NAMA_FAKTA_ANTARA,
    "URGENSI": URGENSI,
    "GOAL_KESELAMATAN": GOAL_KESELAMATAN,
    "JAWABAN_DEMO": JAWABAN_DEMO,
}

filename = "basis_pengetahuan_servis.sav"
with open(filename, "wb") as f:
    pickle.dump(basis_pengetahuan, f)
print(f"Basis pengetahuan disimpan ke '{filename}'")

Basis pengetahuan disimpan ke 'basis_pengetahuan_servis.sav'


In [14]:
# Cek: load ulang file .sav lalu uji dengan skenario demo
with open("basis_pengetahuan_servis.sav", "rb") as f:
    kb = pickle.load(f)
print("Isi file:", list(kb.keys()))

p = kb["PROFIL_MOTOR"][kb["PROFIL_DEMO"]]
uji = BackwardChaining(kb["RULES"], kb["FAKTA_DASAR"], kb["PARAMETER"], p["interval"],
                       PenjawabDiam(kb["JAWABAN_DEMO"]),
                       {k: v for k, v in p["karakter"].items() if v is not None})
hasil = uji.jalankan(list(kb["GOALS"]))
print("Hasil dari file .sav:", [kb["GOALS"][g][0] for g in hasil])

Isi file: ['PROFIL_MOTOR', 'PROFIL_DEMO', 'KARAKTERISTIK', 'LABEL_INTERVAL', 'PARAMETER', 'FAKTA_DASAR', 'RULES', 'GOALS', 'NAMA_FAKTA_ANTARA', 'URGENSI', 'GOAL_KESELAMATAN', 'JAWABAN_DEMO']
Hasil dari file .sav: ['Ganti Oli Mesin', 'Ganti Busi', 'Ganti Filter Udara', 'Servis / Pembersihan CVT', 'Ganti V-Belt', 'Ganti Minyak Rem']


## 8. Menjalankan Aplikasi Web (`app.py`)

Aplikasi interaktif dibuat terpisah di file **`app.py`** menggunakan **Streamlit**. Mesin inferensi yang dipakai sama dengan notebook ini.

**Cara menjalankan** (buka Terminal / Anaconda Prompt di folder projek):
```bash
pip install streamlit
streamlit run app.py
```
Browser akan terbuka otomatis di `http://localhost:8501`.

**Fitur aplikasi:**
- **Pilih profil motor** (Honda, Yamaha, Suzuki, atau motor matic lain) + **sesuaikan interval** sesuai buku pedoman
- **Cek servis tertentu saja**: pilih satu/beberapa goal, sistem hanya menanyakan yang relevan
- **Tingkat urgensi**: SEGERA / PERLU / SEBENTAR LAGI / AMAN
- **Jalankan Demo**: konsultasi berjalan otomatis (cocok untuk presentasi)
- **Jalur penelusuran** (goal → rule → sub-goal → fakta) di setiap pertanyaan
- **Log inferensi LIVE** berwarna
- **Status goal** di sidebar
- Tombol **Kembali** & **Mulai Ulang**
- Hasil: rekomendasi + alasan, proses inferensi, tabel jawaban, dan **unduh laporan .txt**

**Teknik penghubung Streamlit ↔ mesin inferensi:** Streamlit menjalankan ulang script setiap ada klik. Jawaban disimpan di `st.session_state`, lalu inferensi dijalankan dari awal. Saat mesin butuh fakta yang belum dijawab, ia melempar `PerluJawaban` dan aplikasi menampilkan pertanyaan tersebut.

## 9. Kesimpulan

1. Sistem pakar berhasil merekomendasikan **12 jenis servis berkala** motor matic menggunakan **33 rule** dan metode **Backward Chaining**.
2. Sistem **tidak terikat satu tipe motor**: interval servis dan karakteristik diambil dari **profil motor** (7 profil), sehingga jawaban yang sama bisa menghasilkan rekomendasi berbeda sesuai motornya.
3. Backward chaining bekerja **mundur dari goal ke fakta**, sehingga sistem **hanya menanyakan fakta yang relevan**. Karakteristik motor dari profil menjadi *fakta awal*, sehingga rule yang tidak cocok (misal coolant pada motor tanpa radiator) gugur tanpa pertanyaan.
4. Adanya **sub-goal** (misal *indikasi masalah CVT*) membuat satu fakta antara bisa dipakai ulang oleh beberapa goal (servis CVT dan ganti V-belt) tanpa bertanya ulang, berkat *working memory*.
5. Pengguna bisa **memilih goal tertentu saja**, yang menunjukkan sifat *goal-driven* dari backward chaining.
6. *Explanation facility* dan **tingkat urgensi** membuat setiap rekomendasi dapat dipertanggungjawabkan dan diprioritaskan.
7. Hasil pengujian *black box* menunjukkan seluruh skenario menghasilkan rekomendasi yang sesuai dengan harapan.

### Referensi

**A. Jurnal (sistem pakar kendaraan & metode inferensi)**

1. Pratama, A. G., Rizky, R., Yunita, A. M., & Wardah, N. N. (2020). Implementasi Metode Backward Chaining untuk Diagnosa Kerusakan Motor Matic Injection. *Explore: Jurnal Sistem Informasi dan Telematika*, 11(2), 91–96. https://doi.org/10.36448/jsit.v11i2.1515
2. Ilyas, M., Hutagalung, J. E., & Suparmadi. (2022). Penerapan Metode Forward Chaining Pada Sistem Pakar Untuk Diagnosa Kerusakan Sepeda Motor Berbasis Web. *Building of Informatics, Technology and Science (BITS)*, 4(2), 932–942. https://doi.org/10.47065/bits.v4i2.2163
3. Akhsa, A. C. D. (2024). Sistem Pakar Diagnosa Kerusakan Motor Matic Menggunakan Metode Forward Chaining. *MALCOM: Indonesian Journal of Machine Learning and Computer Science*, 4(2), 452–462. https://doi.org/10.57152/malcom.v4i2.1189
4. Ramadhan, E., Rahayu, W., & Haryono, S. (2026). Sistem Pakar Untuk Identifikasi Kerusakan Sepeda Motor Menggunakan Metode Forward Chaining pada Bengkel AKC Workshop. *Jurnal Riset dan Aplikasi Mahasiswa Informatika (JRAMI)*, 7(02), 334–343. https://doi.org/10.30998/jrami.v7i02.824
5. Indriani, N., & Apandi, S. (2026). Sistem Pakar Diagnosis Kerusakan Motor Matic Menggunakan Metode Forward Chaining Pada Bengkel Tri Jaya Motor. *Journal of Artificial Intelligence and Innovative Applications (JOAIIA)*, 7(2), 352–359. https://doi.org/10.32493/joaiia.v7i2.58167
6. Gunawan, C. R., Apriani, W., Maulana, T., & Nurhayati. (2025). Sistem Pakar: Identifikasi Kerusakan Menggunakan Metode Backward Chaining. *Amalgamasi: Journal of Mathematics and Applications*, 4(2), 71–77. https://doi.org/10.55098/amalgamasi.v4.i2.pp71-77
7. Sopian, A., Astuti, R. W., Louis, A., & Antika, R. (2025). Sistem Pakar Diagnosa Kerusakan Sepeda Motor Honda Matic Injeksi dengan Menggunakan Metode Forward Chaining Berbasis Web. *Jurnal Akademika*, 17(2), 20–25. https://doi.org/10.53564/akademika.v17i2.1697
8. Mudinah, Y. K. (2024). Sistem Pakar Diagnosa Kerusakan Kendaraan Bermotor Jenis Matic Menggunakan Metode Naive Bayes. *Jurnal Informatika Kaputama (JIK)*, 8(1), 61–70. https://doi.org/10.59697/jik.v8i1.473
9. Naryanto, R. F., Delimayanti, M. K., Kriswanto, Musyono, A. D. N. I., Sukoco, I., & Aditya, M. N. (2022). Development of a mobile expert system for the diagnosis on motorcycle damage using forward chaining algorithm. *Indonesian Journal of Electrical Engineering and Computer Science*, 27(3), 1601–1609. https://doi.org/10.11591/ijeecs.v27.i3.pp1601-1609
10. Syaputra, A., & Setiadi, D. (2020). Sistem Pakar Diagnosa Kerusakan Sepeda Motor Yamaha Matic Menggunakan Metode Forward Chaining. *Jusikom: Jurnal Sistem Komputer Musirawas*, 5(2), 126–135. https://doi.org/10.32767/jusikom.v5i2.1039

**B. Buku pedoman pemilik (sumber interval servis)**

11. PT Astra Honda Motor. *Buku Pedoman Pemilik Honda Vario 125*.
12. Honda Motor Co., Ltd. *Owner's Manual Honda BeAT (ACH110)*, *Click125 (ACB125)*, dan *ADV160*: Maintenance Schedule.
13. Yamaha Motor Co., Ltd. *Owner's Manual NMAX (GPD150-A)* dan *Gear 125 (LNP125)*: Periodic Maintenance Chart.
14. Suzuki Motor Corporation. *Owner's Manual Address 110 (UK110)*: Maintenance Chart.

**C. Peta pemakaian referensi (untuk penulisan laporan)**

| No | Dipakai untuk bagian | Kaitan dengan ServisPro |
|---|---|---|
| 1 | Landasan metode, penelitian terdahulu | Backward chaining pada motor matic injeksi (paling dekat dengan penelitian ini) |
| 6 | Landasan metode | Backward chaining terbaru (objek: kerusakan komputer) |
| 2, 3, 4, 5, 9 | Penelitian terdahulu, pembanding metode | Sistem pakar kerusakan sepeda motor dengan forward chaining |
| 7 | Penelitian terdahulu, profil Honda | Motor Honda matic injeksi |
| 10 | Penelitian terdahulu, profil Yamaha | Motor Yamaha matic |
| 8 | Pembanding metode | Motor matic dengan Naive Bayes (bukan rule-based) |
| 11, 12 | Basis pengetahuan: interval profil Honda | Jadwal perawatan berkala |
| 13 | Basis pengetahuan: interval profil Yamaha | Jadwal perawatan berkala |
| 14 | Basis pengetahuan: interval profil Suzuki | Jadwal perawatan berkala |

> Catatan: no. 12–14 adalah buku pedoman edisi ekspor (berbahasa Inggris). Cocokkan angkanya dengan buku pedoman edisi Indonesia sebelum dikutip di laporan; nilai bertanda **[bengkel]** bukan dari buku pedoman.
> Perbedaan penelitian ini dengan no. 1–10: fokus pada **rekomendasi servis berkala** (bukan diagnosis kerusakan), mendukung **beberapa merek/tipe motor** lewat profil, dan memberi **tingkat urgensi**.